#Modelo KNN

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, roc_auc_score, precision_score, recall_score, f1_score

In [ ]:
df = pd.read_csv('insurance_claims_limpio.csv')
X = df.drop(columns=['fraud_reported'])
y = df['fraud_reported']

In [ ]:
X_encoded = pd.get_dummies(X, drop_first=True)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.20, stratify=y, random_state=42
)


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11, 15, 21],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

In [ ]:
knn = KNeighborsClassifier()
grid = GridSearchCV(
    estimator=knn,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)
grid.fit(X_train_scaled, y_train)

GridSearchCV(cv=5, estimator=KNeighborsClassifier(), n_jobs=-1,
             param_grid={'metric': ['euclidean', 'manhattan'],
                         'n_neighbors': [3, 5, 7, 9, 11, 15, 21],
                         'weights': ['uniform', 'distance']},
             scoring='f1')

In [ ]:
best_knn = grid.best_estimator_

y_pred_train = best_knn.predict(X_train_scaled)
y_pred_test = best_knn.predict(X_test_scaled)

y_prob_train = best_knn.predict_proba(X_train_scaled)[:, 1]
y_prob_test = best_knn.predict_proba(X_test_scaled)[:, 1]

In [ ]:
metrics = {
    'Set': ['Entrenamiento', 'Prueba'],
    'AUC': [roc_auc_score(y_train, y_prob_train), roc_auc_score(y_test, y_prob_test)],
    'Precision': [precision_score(y_train, y_pred_train), precision_score(y_test, y_pred_test)],
    'Recall': [recall_score(y_train, y_pred_train), recall_score(y_test, y_pred_test)],
    'F1-Score': [f1_score(y_train, y_pred_train), f1_score(y_test, y_pred_test)]
}

df_resultados_knn = pd.DataFrame(metrics)
print("Mejores Hiperparámetros:", grid.best_params_)
print("\n--- Desempeño Modelo KNN ---")
print(df_resultados_knn.round(4))

Mejores Hiperparámetros: {'metric': 'euclidean', 'n_neighbors': 3, 'weights': 'uniform'}

--- Desempeño Modelo KNN ---
             Set     AUC  Precision  Recall  F1-Score
0  Entrenamiento  0.8651     0.7462  0.4899    0.5915
1         Prueba  0.4898     0.2581  0.1633    0.2000


#Regresión logic

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

In [ ]:
lr_model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

In [ ]:
y_pred_train = lr_model.predict(X_train_scaled)
y_pred_test = lr_model.predict(X_test_scaled)

y_prob_train = lr_model.predict_proba(X_train_scaled)[:, 1]
y_prob_test = lr_model.predict_proba(X_test_scaled)[:, 1]

In [ ]:
metrics_lr = {
    'Set': ['Entrenamiento', 'Prueba'],
    'AUC': [roc_auc_score(y_train, y_prob_train), roc_auc_score(y_test, y_prob_test)],
    'Precision': [precision_score(y_train, y_pred_train), precision_score(y_test, y_pred_test)],
    'Recall': [recall_score(y_train, y_pred_train), recall_score(y_test, y_pred_test)],
    'F1-Score': [f1_score(y_train, y_pred_train), f1_score(y_test, y_pred_test)]
}

df_resultados_lr = pd.DataFrame(metrics_lr)
print("--- Desempeño Regresión Logística (Baseline) ---")
print(df_resultados_lr.round(4))

--- Desempeño Regresión Logística (Baseline) ---
             Set     AUC  Precision  Recall  F1-Score
0  Entrenamiento  0.9643     0.7410  0.9394    0.8285
1         Prueba  0.7800     0.5439  0.6327    0.5849
